In [0]:
%run ./00_config

In [0]:
# ============================================================================
# US-1.06: Managed Volume and Landing Directory Creation
# ============================================================================

# 1. Create the Managed Volume inside the Bronze Schema
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS `{catalog_name}`.`{schema_bronze}`.landing_volume
  COMMENT 'Managed landing volume container for holding raw, multi-source incoming CSV data files.';
""")
print(f"📦 Managed volume 'landing_volume' verified inside `{catalog_name}`.`{schema_bronze}`.")

# 2. Create the directories using Databricks Utilities (dbutils)
paths_to_create = [
    path_customers,
    path_accounts,
    path_branches,
    path_transactions,
    path_lookup_codes,
    f"{volume_root_path}/staging"
]

print("\n📁 Creating isolated landing directories...")
for directory_path in paths_to_create:
    dbutils.fs.mkdirs(directory_path)
    print(f"  └─ Created/Verified: {directory_path}")

# 3. Final verification list to show they exist
print("\n🔍 Verification: Listing folders inside the landing volume:")
volume_contents = dbutils.fs.ls(volume_root_path)
for file_info in volume_contents:
    print(f"  🔹 {file_info.path}")


In [0]:
def scan_volume_recursively(folder_path):
    """Recursively walks through directories to find all files and sizes."""
    try:
        # List contents of the current folder path
        contents = dbutils.fs.ls(folder_path)
        
        for item in contents:
            if item.isDir():
                # If it's a directory, dive deeper inside it recursively
                scan_volume_recursively(item.path)
            else:
                # If it's a file, calculate a readable size and print it
                size_kb = item.size / 1024
                print(f"📄 File: {item.path:<80} | Size: {size_kb:>7.2f} KB")
                
    except Exception as e:
        print(f"⚠️ Error scanning path {folder_path}: {str(e)}")

# 2. Execute the recursive auditor starting from your landing volume root
print("====================================================================================================")
print(f"🔎 AUDITING ALL UPLOADED FILES IN VOLUME: {volume_root_path}")
print("====================================================================================================")

scan_volume_recursively(volume_root_path)

print("====================================================================================================")
print("✅ File size scan audit complete. Remember to save the notebook to preserve this output block!")
print("====================================================================================================")


In [0]:
target_file_path = f"{path_customers}/customers.csv"

print("====================================================================")
print(f"📄 READING FIRST 500 CHARACTERS OF: {target_file_path}")
print("====================================================================")

try:
    # dbutils.fs.head reads the beginning of a file as raw plain text
    # We request the first 500 bytes to easily see the header and first few rows
    raw_text_peek = dbutils.fs.head(target_file_path, 500)
    print(raw_text_peek)
    
except Exception as e:
    print(f"❌ Error reading file: {str(e)}")
    print("Please verify the file has been uploaded to the correct folder.")

print("====================================================================")
